# T37 - 同花顺Excel导入财务指标

## 5. 核心逻辑实现

本章实现核心业务逻辑，包括同花顺API数据提取、Excel自动化处理和数据库导入。

## 5.1 环境准备

In [ ]:
# 导入标准库
import os
import sys
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep

# 添加项目路径
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

# 导入第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import psycopg2

# 导入项目配置
from config import (
    PROJECT_NAME,
    get_mysql_bond_engine,
    get_postgres_engine,
    get_postgres_connection,
    get_ths_account,
    CHUNK_SIZE,
    SQL_RETRY_ATTEMPTS,
    SQL_RETRY_DELAY,
    EXCEL_RETRY_ATTEMPTS,
    EXCEL_RETRY_DELAY,
    EXCEL_CALCULATION_WAIT,
    TABLE_WIDE_FORMAT,
    TABLE_LONG_FORMAT,
    THS_FINANCIAL_INDICATORS
)

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"项目: {PROJECT_NAME}")
print("核心逻辑模块加载完成")

## 5.2 模块一：同花顺API批量提取

### 5.2.1 API连接管理

In [ ]:
class THSApiManager:
    """同花顺API管理器"""
    
    def __init__(self):
        """初始化API管理器"""
        self.logged_in = False
        self.current_account = None
        
    def login(self):
        """
        登录同花顺API
        
        Returns:
        --------
        bool : 登录是否成功
        """
        try:
            from iFinDPy import THS_iFinDLogin
            
            account = get_ths_account()
            if account is None:
                logger.error("未配置同花顺API账号")
                return False
                
            result = THS_iFinDLogin(account['username'], account['password'])
            
            if result == 0:
                self.logged_in = True
                self.current_account = account
                logger.info("同花顺API登录成功")
                return True
            else:
                logger.error(f"同花顺API登录失败，错误码: {result}")
                return False
                
        except ImportError:
            logger.error("iFinDPy模块未安装，请安装同花顺iFinD终端")
            return False
        except Exception as e:
            logger.error(f"登录异常: {e}")
            return False
    
    def logout(self):
        """登出API"""
        self.logged_in = False
        self.current_account = None
        logger.info("已登出同花顺API")

# 创建API管理器实例
ths_api = THSApiManager()
print("同花顺API管理器创建完成")

### 5.2.2 发行人代码查询

In [ ]:
class IssuerManager:
    """发行人管理器"""
    
    def __init__(self, pg_engine):
        """
        初始化发行人管理器
        
        Parameters:
        -----------
        pg_engine : SQLAlchemy Engine
            PostgreSQL引擎
        """
        self.pg_engine = pg_engine
        
    def get_all_issuers(self):
        """
        获取所有发行人代码
        
        Returns:
        --------
        DataFrame : 发行人列表
        """
        sql = """
        SELECT 
            max(trade_code) AS trade_code,
            ths_issuer_name_cn_bond
        FROM (
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
            UNION ALL
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
            UNION ALL
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
        ) SQ
        GROUP BY ths_issuer_name_cn_bond
        """
        
        df = pd.read_sql(sql, self.pg_engine)
        logger.info(f"获取到 {len(df)} 个发行人")
        return df
    
    def get_trade_codes(self):
        """
        获取发行人代码列表
        
        Returns:
        --------
        list : 代码列表
        """
        df = self.get_all_issuers()
        return df['trade_code'].tolist()

# 创建发行人管理器
pg_engine = get_postgres_engine()
issuer_manager = IssuerManager(pg_engine)
print("发行人管理器创建完成")

### 5.2.3 财务指标提取

In [ ]:
class FinancialIndicatorExtractor:
    """财务指标提取器"""
    
    def __init__(self):
        """初始化提取器"""
        self.indicators = THS_FINANCIAL_INDICATORS
        
    def build_indicator_string(self):
        """
        构建指标字符串
        
        Returns:
        --------
        str : 指标字符串
        """
        return ';'.join(self.indicators)
    
    def build_param_string(self, dt, dt1):
        """
        构建参数字符串
        
        Parameters:
        -----------
        dt : str
            当前报告期（如20230630）
        dt1 : str
            上期报告期（如2022-12-31）
            
        Returns:
        --------
        str : 参数字符串
        """
        # 根据指标数量构建参数
        params = []
        for i, indicator in enumerate(self.indicators):
            if i < 30:  # 前30个指标使用季度数据
                params.append(f'{dt},100')
            elif i < 60:  # 中间指标使用当前期
                params.append(dt)
            else:  # 后续指标使用上期
                params.append(f'{dt1},100')
                
        return ';'.join(params)
    
    def extract_single(self, trade_code, dt, dt1):
        """
        提取单个发行人的财务指标
        
        Parameters:
        -----------
        trade_code : str
            债券代码
        dt : str
            当前报告期
        dt1 : str
            上期报告期
            
        Returns:
        --------
        DataFrame : 财务指标数据
        """
        try:
            from iFinDPy import THS_BD
            
            indicators = self.build_indicator_string()
            params = self.build_param_string(dt, dt1)
            
            df = THS_BD(trade_code, indicators, params)
            
            if df is None or df.data is None:
                logger.warning(f"提取 {trade_code} 数据为空")
                return pd.DataFrame()
                
            result = df.data
            result['dt'] = dt
            
            return result
            
        except Exception as e:
            logger.error(f"提取 {trade_code} 失败: {e}")
            return pd.DataFrame()
    
    def extract_batch(self, trade_codes, dt, dt1, pg_url, table_name):
        """
        批量提取财务指标并保存到数据库
        
        Parameters:
        -----------
        trade_codes : list
            债券代码列表
        dt : str
            当前报告期
        dt1 : str
            上期报告期
        pg_url : str
            PostgreSQL连接URL
        table_name : str
            目标表名
            
        Returns:
        --------
        int : 成功提取的数量
        """
        success_count = 0
        
        for i, trade_code in enumerate(trade_codes):
            logger.info(f"处理 {i+1}/{len(trade_codes)}: {trade_code}")
            
            df = self.extract_single(trade_code, dt, dt1)
            
            if not df.empty:
                df.dropna(how='all', inplace=True)
                try:
                    df.to_sql(table_name, pg_url, if_exists='append', index=False)
                    success_count += 1
                except Exception as e:
                    logger.error(f"保存 {trade_code} 失败: {e}")
                    
            # 避免API频率限制
            sleep(0.5)
            
        logger.info(f"批量提取完成，成功: {success_count}/{len(trade_codes)}")
        return success_count

# 创建提取器实例
extractor = FinancialIndicatorExtractor()
print("财务指标提取器创建完成")

## 5.3 模块二：Excel自动化处理

### 5.3.1 Excel操作类

In [ ]:
def retry(attempts, delay=30):
    """
    重试装饰器
    
    Parameters:
    -----------
    attempts : int
        最大重试次数
    delay : int
        重试延迟(秒)
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    logger.warning(f"尝试 {i+1}/{attempts} 失败: {e}")
                    if i < attempts - 1:
                        time.sleep(delay)
            raise Exception(f"所有 {attempts} 次尝试均失败")
        return wrapper
    return decorator


class ExcelAutomator:
    """Excel自动化操作类"""
    
    def __init__(self):
        """初始化Excel自动化器"""
        self.Excel = None
        self._init_excel()
        
    def _init_excel(self):
        """初始化Excel应用"""
        try:
            import win32com.client as win32
            self.Excel = win32.client.Dispatch("Excel.Application")
            self.Excel.DisplayAlerts = False
            logger.info("Excel应用初始化成功")
        except ImportError:
            logger.warning("win32com模块未安装，Excel自动化功能不可用")
        except Exception as e:
            logger.error(f"Excel初始化失败: {e}")
            
    @retry(attempts=EXCEL_RETRY_ATTEMPTS, delay=EXCEL_RETRY_DELAY)
    def open_workbook(self, file_path):
        """打开工作簿"""
        return self.Excel.Workbooks.Open(file_path)
    
    @retry(attempts=EXCEL_RETRY_ATTEMPTS, delay=EXCEL_RETRY_DELAY)
    def calculate_worksheet(self, ws):
        """计算工作表"""
        ws.Calculate()
    
    @retry(attempts=EXCEL_RETRY_ATTEMPTS, delay=EXCEL_RETRY_DELAY)
    def save_as_values(self, ws):
        """保存为值"""
        used_range = ws.UsedRange
        used_range.Copy()
        used_range.PasteSpecial(Paste=12)  # xlPasteValues
        ws.Application.CutCopyMode = False
    
    @retry(attempts=EXCEL_RETRY_ATTEMPTS, delay=EXCEL_RETRY_DELAY)
    def save_workbook(self, wb):
        """保存工作簿"""
        wb.Save()
    
    @retry(attempts=EXCEL_RETRY_ATTEMPTS, delay=EXCEL_RETRY_DELAY)
    def close_workbook(self, wb):
        """关闭工作簿"""
        wb.Close()
        
    def quit(self):
        """退出Excel应用"""
        if self.Excel:
            self.Excel.Quit()
            logger.info("Excel应用已退出")

# 创建Excel自动化器
excel_automator = ExcelAutomator()
print("Excel自动化器创建完成")

### 5.3.2 批量日期填充

In [ ]:
class ExcelDateFiller:
    """Excel日期填充器"""
    
    def __init__(self, excel_automator):
        """
        初始化日期填充器
        
        Parameters:
        -----------
        excel_automator : ExcelAutomator
            Excel自动化器
        """
        self.automator = excel_automator
        
    def extract_date_from_filename(self, filename):
        """
        从文件名提取日期
        
        Parameters:
        -----------
        filename : str
            文件名（如20220630.xlsx）
            
        Returns:
        --------
        str : 格式化日期（如2022-06-30）
        """
        try:
            date_str = filename.split('.')[0]
            date_obj = datetime.strptime(date_str, '%Y%m%d')
            return date_obj.strftime('%Y-%m-%d')
        except Exception as e:
            logger.error(f"日期提取失败: {e}")
            return None
    
    def fill_date_column(self, folder_path):
        """
        批量填充日期列
        
        Parameters:
        -----------
        folder_path : str
            Excel文件夹路径
            
        Returns:
        --------
        int : 处理的文件数
        """
        files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]
        processed = 0
        
        for filename in files:
            file_path = os.path.join(folder_path, filename)
            formatted_date = self.extract_date_from_filename(filename)
            
            if formatted_date is None:
                continue
                
            try:
                import win32com.client as win32
                
                wb = self.automator.open_workbook(file_path)
                ws = wb.Worksheets(1)
                
                # 获取最右列
                last_column = ws.Cells(1, ws.Columns.Count).End(win32.constants.xlToLeft).Column
                
                # 填充日期
                ws.Range(
                    ws.Cells(2, last_column),
                    ws.Cells(ws.UsedRange.Rows.Count, last_column)
                ).Value = formatted_date
                
                # 自动填充
                col = last_column - 1
                source_range = ws.Range(ws.Cells(2, col), ws.Cells(2, col))
                destination_range = ws.Range(
                    ws.Cells(2, col),
                    ws.Cells(ws.UsedRange.Rows.Count, col)
                )
                source_range.AutoFill(destination_range, win32.constants.xlFillDefault)
                
                self.automator.save_workbook(wb)
                self.automator.close_workbook(wb)
                
                processed += 1
                logger.info(f"处理 {filename} 完成")
                
            except Exception as e:
                logger.error(f"处理 {filename} 失败: {e}")
                
        return processed

# 创建日期填充器
date_filler = ExcelDateFiller(excel_automator)
print("Excel日期填充器创建完成")

### 5.3.3 批量计算并保存为值

In [ ]:
class ExcelCalculator:
    """Excel计算器"""
    
    def __init__(self, excel_automator):
        """
        初始化计算器
        
        Parameters:
        -----------
        excel_automator : ExcelAutomator
            Excel自动化器
        """
        self.automator = excel_automator
        
    def calculate_and_save(self, folder_path):
        """
        批量计算并保存为值
        
        Parameters:
        -----------
        folder_path : str
            Excel文件夹路径
            
        Returns:
        --------
        int : 处理的文件数
        """
        files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]
        processed = 0
        
        for filename in files:
            file_path = os.path.join(folder_path, filename)
            
            try:
                wb = self.automator.open_workbook(file_path)
                
                # 等待公式计算
                sleep(EXCEL_CALCULATION_WAIT)
                
                # 计算工作表
                self.automator.calculate_worksheet(wb.Worksheets(1))
                
                # 保存为值
                self.automator.save_as_values(wb.Worksheets(1))
                
                # 保存工作簿
                self.automator.save_workbook(wb)
                self.automator.close_workbook(wb)
                
                processed += 1
                logger.info(f"处理 {filename} 完成")
                
            except Exception as e:
                logger.error(f"处理 {filename} 失败: {e}")
                
        return processed

# 创建计算器
calculator = ExcelCalculator(excel_automator)
print("Excel计算器创建完成")

## 5.4 模块三：数据库导入

### 5.4.1 Excel导入器

In [ ]:
class ExcelToPostgresImporter:
    """Excel到PostgreSQL导入器"""
    
    def __init__(self, pg_url):
        """
        初始化导入器
        
        Parameters:
        -----------
        pg_url : str
            PostgreSQL连接URL
        """
        self.pg_url = pg_url
        
    def import_excel_files(self, folder_path, table_name):
        """
        批量导入Excel文件
        
        Parameters:
        -----------
        folder_path : str
            Excel文件夹路径
        table_name : str
            目标表名
            
        Returns:
        --------
        int : 导入的总行数
        """
        files = [f for f in os.listdir(folder_path) 
                 if f.endswith('.xlsx') or f.endswith('.xls')]
        total_rows = 0
        
        for filename in files:
            file_path = os.path.join(folder_path, filename)
            
            try:
                df = pd.read_excel(file_path)
                df.to_sql(table_name, self.pg_url, if_exists='append', index=False)
                total_rows += len(df)
                logger.info(f"导入 {filename}: {len(df)} 行")
                
            except Exception as e:
                logger.error(f"导入 {filename} 失败: {e}")
                
        logger.info(f"总计导入 {total_rows} 行")
        return total_rows
    
    def convert_column_types(self, table_name, exclude_columns=['dt', 'thscode']):
        """
        转换列类型为FLOAT
        
        Parameters:
        -----------
        table_name : str
            表名
        exclude_columns : list
            排除的列
        """
        pg_engine = get_postgres_engine()
        
        # 获取列名
        sql = f"""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = '{table_name}'
        """
        df = pd.read_sql(sql, pg_engine)
        columns = [c for c in df['column_name'].tolist() if c not in exclude_columns]
        
        # 转换类型
        pg_conn = get_postgres_connection()
        cursor = pg_conn.cursor()
        
        for col in columns:
            try:
                cursor.execute(f"""
                    ALTER TABLE {table_name}
                    ALTER COLUMN {col} TYPE FLOAT USING {col}::FLOAT
                """)
            except Exception as e:
                logger.warning(f"转换列 {col} 失败: {e}")
                
        pg_conn.commit()
        cursor.close()
        pg_conn.close()
        
        logger.info("列类型转换完成")

# 创建导入器
from config import POSTGRES_URL
importer = ExcelToPostgresImporter(POSTGRES_URL)
print("Excel导入器创建完成")

### 5.4.2 宽表转长表

In [ ]:
class WideToLongConverter:
    """宽表转长表转换器"""
    
    def __init__(self, pg_url):
        """
        初始化转换器
        
        Parameters:
        -----------
        pg_url : str
            PostgreSQL连接URL
        """
        self.pg_url = pg_url
        
    def convert(self, source_table, target_table, chunksize=CHUNK_SIZE):
        """
        执行宽表到长表转换
        
        Parameters:
        -----------
        source_table : str
            源表名
        target_table : str
            目标表名
        chunksize : int
            分块大小
            
        Returns:
        --------
        int : 处理的总行数
        """
        query = f"SELECT * FROM {source_table}"
        chunks = pd.read_sql(query, self.pg_url, chunksize=chunksize)
        
        processed_rows = 0
        
        for df in chunks:
            # 宽表转长表
            df_long = df.melt(
                id_vars=['dt', 'thscode'],
                var_name='指标',
                value_name='value'
            )
            
            # 重命名
            df_long.rename(columns={'thscode': 'trade_code'}, inplace=True)
            
            # 写入目标表
            try:
                df_long.to_sql(target_table, self.pg_url, if_exists='append', index=False)
            except Exception as e:
                logger.error(f"写入失败: {e}")
                
            processed_rows += len(df)
            logger.info(f"已处理 {processed_rows} 行")
            
        logger.info(f"转换完成，共处理 {processed_rows} 行")
        return processed_rows

# 创建转换器
converter = WideToLongConverter(POSTGRES_URL)
print("宽表转长表转换器创建完成")